In [ ]:

# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATASETS
# TO THE CORRECT LOCATION (/kaggle/input) IN YOUR NOTEBOOK,
# THEN FEEL FREE TO DELETE CELL.
'''
import os
import sys
from tempfile import NamedTemporaryFile
from urllib.request import urlopen
from urllib.parse import unquote
from urllib.error import HTTPError
from zipfile import ZipFile

CHUNK_SIZE = 40960 
DATASET_MAPPING = 'tabular-playground-series-feb-2021:https%3A%2F%2Fstorage.googleapis.com%2Fkaggle-competitions-data%2Fkaggle-v2%2F25225%2F1923495%2Fbundle%2Farchive.zip%3FX-Goog-Algorithm%3DGOOG4-RSA-SHA256%26X-Goog-Credential%3Dgcp-kaggle-com%2540kaggle-161607.iam.gserviceaccount.com%252F20210208%252Fauto%252Fstorage%252Fgoog4_request%26X-Goog-Date%3D20210208T152806Z%26X-Goog-Expires%3D259199%26X-Goog-SignedHeaders%3Dhost%26X-Goog-Signature%3Daa7afbe87c577f16206b41ef193717b54b23ff6ba665e6f18f4d03f107f08c34d3ad6f5afa301d1b0f289b6d82a9101c38ab20a8fa96346fefb5446bd298f44daac2c5b38eaaa52644635d0dcea55f6beb935a1f575478f425a15e0cf0dd9434ff9962355512a40b5504a12e5a70d636276a611c8ee194987c6e8ded3b36877f6c0875afd75333b0df4d1fdf6051eb1711f7779f4f6542360cb860b6e6e9afa88a51c27cabf129a460f6f1a72850e889232370ef0ee1f2513cb44d47cc87faa6204be8ddc5b4e2d9fd28b5dcf704ef5e65bdd969dc18d0079586d8c79c12e65057a49827978d009ed6797670d2a28f2174aba5fcbfdf9db7306d59f50d11da8d'
KAGGLE_INPUT_PATH='/home/kaggle/input'
KAGGLE_INPUT_SYMLINK='/kaggle'

os.makedirs(KAGGLE_INPUT_PATH, 777)
os.symlink(KAGGLE_INPUT_PATH, os.path.join('..', 'input'), target_is_directory=True)
os.makedirs(KAGGLE_INPUT_SYMLINK)
os.symlink(KAGGLE_INPUT_PATH, os.path.join(KAGGLE_INPUT_SYMLINK, 'input'), target_is_directory=True)

for dataset_mapping in DATASET_MAPPING.split(','):
    directory, download_url_encoded = dataset_mapping.split(':')
    download_url = unquote(download_url_encoded)
    destination_path = os.path.join(KAGGLE_INPUT_PATH, directory)
    try:
        with urlopen(download_url) as zipfileres, NamedTemporaryFile() as tfile:
            total_length = zipfileres.headers['content-length']
            print(f'Downloading {directory}, {total_length} bytes zipped')
            dl = 0
            data = zipfileres.read(CHUNK_SIZE)
            while len(data) > 0:
                dl += len(data)
                tfile.write(data)
                done = int(50 * dl / int(total_length))
                sys.stdout.write(f"\r[{'=' * done}{' ' * (50-done)}] {dl} bytes downloaded")
                sys.stdout.flush()
                data = zipfileres.read(CHUNK_SIZE)
            print(f'\nUnzipping {directory}')
            with ZipFile(tfile) as zfile:
                zfile.extractall(destination_path)
    except HTTPError as e:
        print(f'Failed to load (likely expired) {download_url} to path {destination_path}')
        continue
    except OSError as e:
        print(f'Failed to load {download_url} to path {destination_path}')
        continue
print('Dataset import complete.')
'''

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train_df = pd.read_csv('../input/tabular-playground-series-feb-2021/train.csv')
test_df = pd.read_csv('../input/tabular-playground-series-feb-2021/test.csv')

In [ ]:
train_df.head(5)

In [ ]:
train_df.describe()

# Checking Missing Values

In [ ]:
train_df.isnull().sum()

# Unique values for Categorical Columns

In [ ]:
def get_unique_val(df, col):
    return df[col].unique()

In [ ]:
# Training data
for i in range(0,10):
    print('Unique values of cat'+ str(i) + ': ')
    print(get_unique_val(train_df, 'cat'+ str(i)))

In [ ]:
# Test Data
for i in range(0,10):
    print('Unique values of cat'+ str(i) + ': ')
    print(get_unique_val(test_df, 'cat'+ str(i)))

In [ ]:
for i in range(0,10):
    train_minus_test = set(get_unique_val(train_df, 'cat'+ str(i))).difference(set(get_unique_val(test_df, 'cat'+ str(i))))
    test_minus_train = set(get_unique_val(test_df, 'cat'+ str(i))).difference(set(get_unique_val(train_df, 'cat'+ str(i))))
    print('-----Train - Test-----')
    print(train_minus_test)
    print('-----Test - Train-----')
    print(test_minus_train)
    if len(train_minus_test) > 0:
        for item in train_minus_test:
            train_df = train_df[train_df['cat'+ str(i)] != item]
    
    if len(test_minus_train) > 0:
        for item in test_minus_train:
            test_df = test_df[test_df['cat'+ str(i)] != item]

In [ ]:
# Training data after removal of extra records that were present in either of training data and testing data but absent in another one
for i in range(0,10):
    print('Unique values of cat'+ str(i) + ': ')
    print(get_unique_val(train_df, 'cat'+ str(i)))

In [ ]:
# Test Data after removal of extra records that were present in either of training data and testing data but absent in another one
for i in range(0,10):
    print('Unique values of cat'+ str(i) + ': ')
    print(get_unique_val(test_df, 'cat'+ str(i)))

In [ ]:
train_df.count()

In [ ]:
test_df.count()

Creating a dataframe of categorical columns only

In [ ]:
object_cols = train_df.select_dtypes(include=['object'])
object_cols.head(5)

# One - hot encoding
One - hot encoding does not work well if any categorical column has a lot of categorical values because it will increase size of the data. Hence, we will consider columns with upto 10 categorical values, rest categorical values will be dropped.

In [ ]:
low_cardinality_cols = [col for col in object_cols if train_df[col].nunique()<10]
low_cardinality_cols

In [ ]:
high_cardinality_cols = list(set(object_cols) - set(low_cardinality_cols))
high_cardinality_cols

In [ ]:
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder = OneHotEncoder(handle_unknown = 'ignore', sparse = False)
one_hot_encoded_cols = pd.DataFrame(one_hot_encoder.fit_transform(train_df[low_cardinality_cols]))
one_hot_encoded_cols.index = train_df.index

In [ ]:
one_hot_encoded_cols.head(5)

In [ ]:
one_hot_encoded_test_cols = pd.DataFrame(one_hot_encoder.fit_transform(test_df[low_cardinality_cols]))
one_hot_encoded_test_cols.index = test_df.index

In [ ]:
one_hot_encoded_test_cols.head(5)

Separating numeric columns

In [ ]:
numeric_train_df = train_df.drop(object_cols, axis = 1)

In [ ]:
numeric_train_df.head(5)

In [ ]:
numeric_test_df = test_df.drop(object_cols, axis = 1)

In [ ]:
numeric_test_df.head(5)

Combining one hot encoded categorical and numerical data

In [ ]:
final_train_df = pd.concat([one_hot_encoded_cols, numeric_train_df], axis = 1)

In [ ]:
final_train_df.head(5)

In [ ]:
final_test_df = pd.concat([one_hot_encoded_test_cols, numeric_test_df], axis = 1)

In [ ]:
final_test_df.head(5)

In [ ]:
final_train_df.columns

In [ ]:
final_test_df.columns

# Splitting of training data into training and validation data

In [ ]:
y = final_train_df.pop('target')
y.head(5)

In [ ]:
X = final_train_df
X.head(5)

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(X, y, random_state = 0)

In [ ]:
x_train.head(5)

In [ ]:
x_val.head(5)

# Model and training

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

model = RandomForestRegressor(n_estimators = 100, random_state = 0)
model.fit(x_train, y_train)
pred = model.predict(x_val)
rmse = mean_squared_error(y_val, pred, squared = False)
rmse

# Prediction

In [ ]:
final_test_df.head(5)

In [ ]:
predictions = model.predict(final_test_df)
predictions

# Submission

In [ ]:
result = pd.DataFrame()
result['id'] = final_test_df['id']
result['target'] = predictions
result.head(5)

In [ ]:
result.to_csv('../output/submission.csv', index = False)

In [ ]:
!kaggle competitions submit -c tabular-playground-series-feb-2021 -f submission.csv -m "Message"